# Getting Started with Automated-LLM-Probes

In [1]:
from __future__ import annotations
import os
from pathlib import Path
import automated_intelligence_tests as ait
import automated_llm_probes as alp
assert os.environ.get("OPENAI_API_KEY")

### Models available

In [2]:
models = alp.ready_models()
print(f"|Models available: {len(models)}|\n")
for m in models:
    print(f"  {m['name']:20s} {m['api']:12s} {m['model_id']}")

|Models available: 57|

  GPT-3.5-Turbo        openai       gpt-3.5-turbo
  GPT-4.0-Turbo        openai       gpt-4-turbo-2024-04-09
  Moonshot-v1-8k       moonshot     moonshot-v1-8k
  Moonshot-v1-128k     moonshot     moonshot-v1-128k
  Claude-3.5-Sonnet    claude       claude-3.5-sonnet
  GPT-4o               openai       gpt-4o-2024-08-06
  DeepSeek-R1          deepseek     deepseek-r1
  Qwen-Turbo           qwen         qwen-turbo
  o4-mini              openai       o4-mini-2025-04-16
  GPT-4.1              openai       gpt-4.1-2025-04-14
  GPT-4.1-mini         openai       gpt-4.1-mini-2025-04-14
  GPT-4.1-nano         openai       gpt-4.1-nano-2025-04-14
  Kimi-K2              moonshot     moonshot-v1-32k
  Qwen3-235B-Instruct  qwen         qwen3-235b-a22b-instruct-2507
  GPT-5                openai       gpt-5-2025-08-07
  GPT-5-mini           openai       gpt-5-mini-2025-08-07
  Claude-Opus-4.1      claude       claude-opus-4-1-20250805
  Grok-Code-Fast       spacexai     grok

### Probed tasks

In [3]:
path = Path('./data/'); n=1
print(f"|Probed tasks: {len([p for p in path.iterdir() if p.is_dir()])}|\n")
for d in path.iterdir():
    if d.is_dir():
        count = sum(1 for f in d.rglob('*') if f.is_file())
        print(f"  {n}. {d.name.upper()[:7]:8}:  {count}"); n+=1

|Probed tasks: 3|

  1. AUT     :  5493
  2. DAT     :  5190
  3. WRT     :  5315


### Tests availabe:

In [4]:
ait_counts = ait.list_available_tests()
print(f'|Available tests: {len(ait_counts)}|\n')
for i,v in ait_counts.items():
    print(f'  {i} : {v}')

|Available tests: 4|

  AUT : Alternative Uses Task
  CAT : Convergent Association Task
  DAT : Divergent Association Task
  WRT : Creative Writing Task


## Tiny trial collection (DAT, 2 responses)

In [5]:
def test_model(test_name="DAT", model_name="GPT-3.5-Turbo", n=250):
    models = [m for m in alp.ready_models() if m["name"] == model_name]
    alp.collect(test_name, models=models, n_per_model=n)

test_model("DAT", n=265)

  GPT-3.5-Turbo: 265/265 done — skip


## Parse & merge

In [20]:
from __future__ import annotations
import re, pandas as pd
import glove_word_embeddings as gwe

def parse_dat(raw):
    text = str(raw or "").strip().strip('"').strip("'")
    tokens = re.split(r"[,\n\r]+", text)
    nouns = [n for n in (gwe.pre.clean_word(t) for t in tokens) if n][:10]
    return nouns + [""] * (10 - len(nouns))

def parse_aut(raw):
    uses = []
    for line in re.split(r"[\n\r]+", str(raw or "")):
        line = re.sub(r"^\s*[\d\.\)\-]+\s*", "", line)
        if toks := [t for t in (gwe.pre.clean_word(t) for t in line.split()) if t]:
            uses.append(" ".join(toks))
    return ", ".join(uses)

def parse_wrt(raw):
    text = re.sub(r"^#+\s*.*$", "", str(raw or ""), flags=re.M)
    text = re.sub(r"^\s*Title:.*$", "", text, flags=re.M | re.I)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def load_task(task: str) -> pd.DataFrame:
    task = task.lower()
    df = pd.DataFrame.from_dict(alp.parse_and_merge(task),orient='index')
    if task == "dat":
        parsed = df["raw"].map(parse_dat)
        df[[f"noun_{i}" for i in range(10)]] = parsed.tolist()
        df["response_clean"] = parsed.map(lambda x: ", ".join(n for n in x if n))
        extra = [f"noun_{i}" for i in range(10)]
    elif task == "aut":
        df["object"] = df["prompt"].str.extract(
            r"object: (.+?)\?", expand=False).str.strip()
        df["response_clean"] = df["raw"].map(parse_aut)
        extra = ["object"]
    elif task == "wrt":
        cues = df["prompt"].str.extract(
            r"words: (.+?)\.", expand=False).str.strip().str.split(r",\s*")
        df[["cue_0", "cue_1", "cue_2"]] = pd.DataFrame(cues.tolist()).iloc[:, :3]
        df["response_clean"] = df["raw"].map(parse_wrt)
        extra = ["cue_0", "cue_1", "cue_2"]
    else:
        raise ValueError(f"Unknown task: {task}")
    cols = ["task", "model_name", "model_id", 
            "provider", "rep", "temperature_std"] + extra + [
        "prompt", "response_clean", "ts_utc", "hash"]
    return df[[c for c in cols if c in df.columns]].sort_values(
        ["model_name", "rep"]).reset_index(drop=True)

def load_tasks():
    for task in ("dat", "aut", "wrt"):
        print(f"Parsing {task.upper()}...")
        df = load_task(task)
        print(df.shape)
        df.to_csv(f"./data/{task}.csv", index=False)

load_task('dat')

dat: 100%|██████████████████████████████████████████████████████████| 5190/5190 [00:08<00:00, 600.38it/s]


,task,model_name,model_id,provider,rep,temperature_std,noun_0,noun_1,noun_2,noun_3,noun_4,noun_5,noun_6,noun_7,noun_8,noun_9,prompt,response_clean,ts_utc
0,DAT,Claude Haiku 4.5,claude-haiku-4-5-20251001,anthropic,0,0.5,telescope,butterfly,courage,sandwich,thunder,algorithm,shadow,marble,whisper,glacier,Generate 10 nouns that are as different from e...,"telescope, butterfly, courage, sandwich, thund...",2026-08-17T10:20:06.301758+00:00
1,DAT,Claude Haiku 4.5,claude-haiku-4-5-20251001,anthropic,1,0.5,telescope,butterfly,democracy,thunder,sandwich,jealousy,volcano,whisper,algorithm,feather,Generate 10 nouns that are as different from e...,"telescope, butterfly, democracy, thunder, sand...",2026-08-17T10:20:07.429834+00:00
2,DAT,Claude Haiku 4.5,claude-haiku-4-5-20251001,anthropic,2,0.5,telescope,melody,courage,volcano,butterfly,democracy,sandwich,shadow,algorithm,ocean,Generate 10 nouns that are as different from e...,"telescope, melody, courage, volcano, butterfly...",2026-08-17T10:20:08.590234+00:00
3,DAT,Claude Haiku 4.5,claude-haiku-4-5-20251001,anthropic,3,0.5,telescope,butterfly,democracy,thunder,sandwich,courage,river,marble,symphony,shadow,Generate 10 nouns that are as different from e...,"telescope, butterfly, democracy, thunder, sand...",2026-08-17T10:20:09.626628+00:00
4,DAT,Claude Haiku 4.5,claude-haiku-4-5-20251001,anthropic,4,0.5,telescope,courage,sandwich,hurricane,melody,shadow,marble,jealousy,algorithm,feather,Generate 10 nouns that are as different from e...,"telescope, courage, sandwich, hurricane, melod...",2026-08-17T10:20:10.846696+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5050,DAT,Llama-4 Scout,meta-llama/llama-4-scout,openrouter,245,0.5,cloud,chair,ocean,smile,house,mountain,library,piano,sandwich,butterfly,Generate 10 nouns that are as different from e...,"cloud, chair, ocean, smile, house, mountain, l...",2026-08-19T01:10:49.777919+00:00
5051,DAT,Llama-4 Scout,meta-llama/llama-4-scout,openrouter,246,0.5,cloud,chair,ocean,smile,book,mountain,kitchen,guitar,flower,robot,Generate 10 nouns that are as different from e...,"cloud, chair, ocean, smile, book, mountain, ki...",2026-08-19T01:10:51.004805+00:00
5052,DAT,Llama-4 Scout,meta-llama/llama-4-scout,openrouter,247,0.5,cloud,chair,river,smile,fungus,library,kite,sandwich,mountain,pillow,Generate 10 nouns that are as different from e...,"cloud, chair, river, smile, fungus, library, k...",2026-08-19T01:10:51.972034+00:00
5053,DAT,Llama-4 Scout,meta-llama/llama-4-scout,openrouter,248,0.5,cloud,chair,ocean,smile,book,mountain,kitchen,guitar,flower,robot,Generate 10 nouns that are as different from e...,"cloud, chair, ocean, smile, book, mountain, ki...",2026-08-19T01:10:52.880622+00:00
